# DuckPD Advanced Features & New Capabilities Walkthrough

Welcome to the **DuckPD Features Walkthrough**! This notebook demonstrates the newest capabilities of DuckPD on **real-world financial market data** using the [AlphaDojo/dojo_stock_news](https://huggingface.co/datasets/AlphaDojo/dojo_stock_news) dataset (~3.9M articles).

### What you will see:
- **Direct Remote Parquet Scanning**: Query millions of rows in cloud parquet without loading full datasets into Python memory.
- **Vectorized String Accessors (`.str`)**: Clean publisher names, extract headlines, and filter topics lazily.
- **Multi-Table Relational Merges (`merge`)**: Join multi-million row news feeds with ticker metadata tables.
- **Extended Reductions**: Compute standard deviation (`std`), variance (`var`), median (`median`), and quantiles (`quantile`).
- **Multi-Frame Concatenation (`duckpd.concat`)**: Combine filtered partitions with automatic schema union and null-padding.
- **Advanced Multi-Column GroupBy**: Named aggregations across publishers and tickers.
- **Transparent Query Plans (`explain()`) & Zero-Copy Parquet Export**: Pushdown inspection and direct disk writes.

## 1. Setup Session & Connect to Remote Parquet

Initialize a DuckPD session with custom memory and execution settings, then lazily scan the 3.9M row dataset hosted on Hugging Face.

In [ ]:
import pandas as std_pd

import duckpd as pd

print(f"DuckPD Version: {pd.__version__}")
session = pd.connect(memory_limit="1GB", threads=4)

# Remote dataset from AlphaDojo (~3.9M financial news rows)
DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"

# Lazily scan parquet directly over HTTP without downloading whole file into memory
news_df = session.read_parquet(DATA_URL)

print("Lazy DataFrame created:")
print(f"Columns: {news_df.columns}")
print(f"Session executions so far: {session.execution_count}")

DuckPD Version: 0.0.4
Lazy DataFrame created:
Columns: ('title', 'image', 'ago', 'primarysymbol', 'primarytopic', 'publisher', 'url', 'id', 'imagedomain', 'description', 'primarytopic_url', 'publisher_logo', 'publish_date', 'on_symbol_json', 'symbol', 'source')
Session executions so far: 0


## 2. Vectorized String Accessors (`.str`)

Clean publisher names, compute headline lengths, and flag earnings-related announcements lazily using DuckPD's `.str` accessor methods.

In [36]:
# Perform lazy string feature engineering
enriched_news = news_df.assign(
    publisher_clean=news_df["publisher"].str.strip().str.upper(),
    title_len=news_df["title"].str.len(),
    is_earnings=news_df["title"].str.upper().str.contains("EARNINGS"),
    is_option_activity=news_df["title"].str.contains("Option Activity"),
)

# Inspect a bounded preview pushed down to DuckDB
enriched_news[["symbol", "publisher_clean", "title_len", "is_earnings", "title"]].head(
    5
)

,symbol,publisher_clean,title_len,is_earnings,title
0,LNGX,BNK INVEST,50,False,"Noteworthy Thursday Option Activity: ZS, MYE, ..."
1,SACHPA,ZACKS,54,False,Neoclouds CRWV & NBIS Soar (Why they have room...
2,SACHPA,ZACKS,64,True,"LQDA Q2 Earnings Top, Strong Yutrepia Sales Fu..."
3,SACHPA,ZACKS,68,True,Allogene Therapeutics' Q2 Earnings Beat Estima...
4,SACHPA,ZACKS,57,False,Can Asceniv Help ADMA Stock Combat IG Market C...


## 3. Multi-Table Relational Merging (`merge`)

Join the multi-million row news dataset with a ticker reference metadata table. The join and predicates are compiled into relational SQL execution.

In [37]:
# Reference table for prominent tech & consumer market cap leaders
ticker_meta = session.from_pandas(
    std_pd.DataFrame(
        {
            "symbol": ["AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "GOOGL"],
            "company_name": [
                "Apple Inc.",
                "NVIDIA Corp.",
                "Microsoft Corp.",
                "Amazon.com Inc.",
                "Tesla Inc.",
                "Alphabet Inc.",
            ],
            "sector": [
                "Technology",
                "Semiconductors",
                "Software",
                "E-Commerce",
                "Automotive",
                "Communication",
            ],
            "market_tier": [
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
            ],
        }
    )
)

# Merge ticker metadata with news stream
news_with_sector = ticker_meta.merge(enriched_news, on="symbol", how="inner")

news_with_sector[
    ["symbol", "company_name", "sector", "publisher_clean", "title_len", "title"]
].head(5)

,symbol,company_name,sector,publisher_clean,title_len,title
0,AAPL,Apple Inc.,Technology,THE MOTLEY FOOL,72,MP Materials Just Signed a Secret Aerospace De...
1,AAPL,Apple Inc.,Technology,RTTNEWS,72,Apple's IPhone 18 Pro And Foldable Ultra Could...
2,AAPL,Apple Inc.,Technology,NASDAQ.COM,109,"After Hours Most Active for Aug 13, 2026 : NX..."
3,AAPL,Apple Inc.,Technology,THE MOTLEY FOOL,69,Amazon Is the Only $3 Trillion Company That Ha...
4,AAPL,Apple Inc.,Technology,THE MOTLEY FOOL,80,SpaceX vs. MP Materials: 2 Very Different Ways...


## 4. Multi-Frame Concatenation (`duckpd.concat`)

Combine distinct ticker news subsets row-wise with automatic schema union and null-padding.

In [38]:
# Split subsets and enrich one partition with custom category tags
nvda_news = news_with_sector[news_with_sector["symbol"] == "NVDA"].assign(
    focus_area="AI Hardware"
)[["symbol", "company_name", "focus_area", "publisher_clean", "title"]]

tsla_news = news_with_sector[news_with_sector["symbol"] == "TSLA"][
    ["symbol", "company_name", "publisher_clean", "title"]
]

# Concatenate partitions: focus_area will be padded with NULLs for TSLA
combined_stream = pd.concat([nvda_news, tsla_news])
print("Union Columns:", combined_stream.columns)

combined_stream.head(6)

Union Columns: ('symbol', 'company_name', 'focus_area', 'publisher_clean', 'title')


,symbol,company_name,focus_area,publisher_clean,title
0,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,Amazon Is the Only $3 Trillion Company That Ha...
1,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,The AI Stock Nobody Is Talking About - but Cou...
2,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,Sam Altman Is Pushing for a $1 Trillion IPO Va...
3,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,This Tech-Focused Vanguard Fund Is Crushing th...
4,NVDA,NVIDIA Corp.,AI Hardware,BARCHART,Stocks Close Higher on Favorable CPI Report an...
5,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,These 3 Tech Stocks Made Double-Digit Dividend...


## 5. Extended Statistical & Boolean Reductions

Calculate statistical metrics across headline length and content properties (`mean`, `median`, `std`, `var`, `quantile`, `any`, `all`) computed in a single SQL query in DuckDB.

In [39]:
print("--- Headline Length Statistical Metrics ---")
print(f"Mean Length:       {news_with_sector['title_len'].mean():.2f}")
print(f"Median Length:     {news_with_sector['title_len'].median():.2f}")
print(f"Std Deviation:     {news_with_sector['title_len'].std():.2f}")
print(f"Variance:          {news_with_sector['title_len'].var():.2f}")
print(f"25th Percentile:   {news_with_sector['title_len'].quantile(0.25):.2f}")
print(f"75th Percentile:   {news_with_sector['title_len'].quantile(0.75):.2f}")
print(f"95th Percentile:   {news_with_sector['title_len'].quantile(0.95):.2f}")

print("\n--- Boolean Reductions on Filtered Subset ---")
print(f"All headlines mention earnings? {news_with_sector['is_earnings'].all()}")
print(f"Any headline mentions earnings? {news_with_sector['is_earnings'].any()}")

--- Headline Length Statistical Metrics ---
Mean Length:       75.86
Median Length:     69.00
Std Deviation:     28.10
Variance:          789.63
25th Percentile:   57.00
75th Percentile:   90.00
95th Percentile:   129.05

--- Boolean Reductions on Filtered Subset ---
All headlines mention earnings? False
Any headline mentions earnings? True


## 6. Advanced GroupBy & Multi-Metric Aggregations

Perform analytical grouping across publishers and tickers using named aggregations, calculating article volume, average length, dispersion, and extreme values.

In [40]:
# Aggregate news analytics by publisher across top market-cap tickers
publisher_analytics = (
    news_with_sector.groupby(["publisher_clean"], as_index=False)
    .agg(
        article_count=("title", "count"),
        avg_headline_len=("title_len", "mean"),
        std_headline_len=("title_len", "std"),
        max_headline_len=("title_len", "max"),
        min_headline_len=("title_len", "min"),
    )
    .sort_values("article_count", ascending=False)
)

publisher_analytics.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len
0,THE MOTLEY FOOL,733,84.780355,30.565115,195,25
1,ZACKS,305,62.875410,13.650148,112,29
2,MARKETBEAT,54,60.666667,11.451291,83,33
3,BARCHART,46,58.978261,10.087372,85,37
4,BNK INVEST,34,42.882353,13.849329,76,23
5,RTTNEWS,15,66.266667,20.026649,95,29
6,NASDAQ.COM,13,97.692308,15.601200,113,74


## 7. Plan Inspection (`explain()`) & Parquet Export (`write_parquet()`)

Inspect the relational query plan generated by DuckPD showing predicate pushdowns, hash joins, and streaming scans into DuckDB, then write the aggregated results directly into Parquet without materializing in Python memory.

In [41]:
print("=== Compiled Query Plan ===")
print(publisher_analytics.explain())

# Export the analytical summary directly to Parquet
publisher_analytics.write_parquet("stock_news_summary.parquet", overwrite=True)
print("\nSuccessfully exported stock_news_summary.parquet directly via DuckDB!")

=== Compiled Query Plan ===


DuckPD logical plan:
SortPlan(input=AggregatePlan(input=JoinPlan(left=ScanPlan(source=PandasSource(key='44b6b88d4cf145de81cf4de7ded57579'), metadata=FrameMetadata(columns=(Column(id=ColumnId(value=UUID('56d0ca61-969d-4740-9857-45f0637486e7')), label='symbol', duckdb_type='VARCHAR', hidden=False), Column(id=ColumnId(value=UUID('b1daa720-3b98-4d89-8798-7091a8874b09')), label='company_name', duckdb_type='VARCHAR', hidden=False), Column(id=ColumnId(value=UUID('dd53acb1-11e2-4380-b03a-9d2beb1845e2')), label='sector', duckdb_type='VARCHAR', hidden=False), Column(id=ColumnId(value=UUID('a175f8a4-5bc1-409a-b398-1a5ad7b6016d')), label='market_tier', duckdb_type='VARCHAR', hidden=False)), index=IndexSpec(columns=(), drop=True, uniqueness=<IndexUniqueness.UNKNOWN: 'unknown'>), ordering=OrderSpec(keys=()))), right=ProjectPlan(input=ProjectPlan(input=ProjectPlan(input=ProjectPlan(input=ScanPlan(source=ParquetSource(paths=('https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.